# HW2 – Factor Models

**Kenza Bouasker**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
import statsmodels.api as sm

plt.rcParams["figure.figsize"] = (9, 5)

## (a) Data

### Stocks & index

In [ ]:
tickers = ["AAPL", "MSFT", "AMZN", "INTC", "JPM", "BAC", "XOM", "CVX",
           "JNJ", "PFE", "PG", "KO", "WMT", "DIS", "CAT"]

px = yf.download(tickers + ["^GSPC"], start="2012-01-01", end="2022-01-01",
                 auto_adjust=True, progress=False)["Close"]
px = px.rename(columns={"^GSPC": "SPX"})
px.tail()

### Returns, train / test split

In [ ]:
rets = px.pct_change().dropna()

train = rets.loc[:"2019-12-31"]
test = rets.loc["2020-01-01":]
len(train), len(test)

### Fama-French factors

- `F-F_Research_Data_Factors_daily.csv`: columns `Mkt-RF, SMB, HML, RF` (header on line 5)
- `F-F_Momentum_Factor_daily.csv`: column `Mom` (header on line 14)

Dates are `YYYYMMDD`, values are in **%**, and each file ends with a copyright line.

In [ ]:
def load_ff(path, skip):
    df = pd.read_csv(path, skiprows=skip, index_col=0)
    df = df[df.index.astype(str).str.strip().str.match(r"^\d{8}$")]   # drop footer
    df.index = pd.to_datetime(df.index.astype(str).str.strip(), format="%Y%m%d")
    df.columns = df.columns.str.strip()
    return df.astype(float) / 100                                      # % -> decimal

ff3 = load_ff("F-F_Research_Data_Factors_daily.csv", 4)
mom = load_ff("F-F_Momentum_Factor_daily.csv", 13)

factors = ff3.join(mom, how="inner").loc["2012":"2021"]
factors.head()

In [ ]:
# align everything on common dates, market = S&P 500 excess return
data = rets.join(factors, how="inner")
data["MKT"] = data["SPX"] - data["RF"]
excess = data[tickers].sub(data["RF"], axis=0)

is_train = data.index <= "2019-12-31"
FF3 = ["MKT", "SMB", "HML"]
CARHART = FF3 + ["Mom"]

## (b) Fama-French 3

$$r_{i,t} - r_{f,t} = \alpha_i + \beta_i\,(r_{m,t} - r_{f,t}) + s_i\,SMB_t + h_i\,HML_t + \varepsilon_{i,t}$$

with $r_m$ the S&P 500 return. Fit on the training set.

### Regressions

In [ ]:
def fit_factors(cols):
    X = sm.add_constant(data.loc[is_train, cols])
    out = {}
    for t in tickers:
        res = sm.OLS(excess.loc[is_train, t], X).fit()
        out[t] = {**res.params, "R2": res.rsquared}
    return pd.DataFrame(out).T

ff_coef = fit_factors(FF3)
ff_coef.round(4)

### Security market line

In [ ]:
def plot_sml(beta, title):
    mu = excess.loc[is_train].mean() * 252               # annualised
    mkt = data.loc[is_train, "MKT"].mean() * 252
    b1, b0 = np.polyfit(beta, mu, 1)
    x = np.linspace(0, beta.max() * 1.15, 50)

    plt.scatter(beta, mu, label="Stocks")
    for t in tickers:
        plt.annotate(t, (beta[t], mu[t]), fontsize=8, xytext=(3, 3), textcoords="offset points")
    plt.scatter(1, mkt, marker="*", s=150, color="k", label="Market")
    plt.plot(x, mkt * x, label=f"Hypothetical: {mkt:.3f}·β")
    plt.plot(x, b0 + b1 * x, "--", label=f"Empirical: {b0:.3f} + {b1:.3f}·β")
    plt.xlabel("β (market)"); plt.ylabel("Mean excess return (annualised)")
    plt.title(title); plt.legend(); plt.grid(alpha=0.3); plt.show()

plot_sml(ff_coef["MKT"], "SML – Fama-French 3 (train)")

**Comments.**

## (c) Carhart 4

$$r_{i,t} - r_{f,t} = \alpha_i + \beta_i\,(r_{m,t} - r_{f,t}) + s_i\,SMB_t + h_i\,HML_t + m_i\,MOM_t + \varepsilon_{i,t}$$

### Regressions

In [ ]:
ca_coef = fit_factors(CARHART)
ca_coef.round(4)

### Security market line

In [ ]:
plot_sml(ca_coef["MKT"], "SML – Carhart 4 (train)")

**Comments.**

## (d) Ex-ante forecasts

$$\hat r_{i,t} - r_{f,t} = \hat\alpha_i + \hat{\boldsymbol\beta}_i^\top \mathbf f_{t-1}$$

Coefficients estimated on the training set, $\text{RMSE}_i = \sqrt{\tfrac1T\sum_t (r_{i,t} - \hat r_{i,t})^2}$ on train and test.

### Predictions

In [ ]:
lagged = data[CARHART].shift(1).iloc[1:]       # f_{t-1}
y = excess.iloc[1:]
tr = lagged.index <= "2019-12-31"

def forecast_rmse(cols):
    X = sm.add_constant(lagged[cols])
    rows = {}
    for t in tickers:
        res = sm.OLS(y.loc[tr, t], X[tr]).fit()
        err = y[t] - res.predict(X)
        rows[t] = {"train": np.sqrt((err[tr] ** 2).mean()),
                   "test": np.sqrt((err[~tr] ** 2).mean())}
    return pd.DataFrame(rows).T

rmse = pd.concat({"FF3": forecast_rmse(FF3), "Carhart": forecast_rmse(CARHART)}, axis=1)
rmse.round(5)

### RMSE – train vs test

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
for ax, m in zip(axes, ["FF3", "Carhart"]):
    rmse[m].plot.bar(ax=ax, title=m, width=0.8)
    ax.set_ylabel("RMSE (daily)"); ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.show()

rmse.mean().unstack().round(5)

**Comments.**